# repo-rag — Colab demo

RAG pipeline over any GitHub repository. This notebook clones this repo, ingests a corpus into ChromaDB, and runs example queries against Claude. The default corpus is [Babel](https://github.com/NFAsylum/babel-tcc); change the `--repo` flag in the ingest cell to point it anywhere.

You need an Anthropic API key (get credits at [console.anthropic.com](https://console.anthropic.com)). The cell below prompts for it with `getpass` so it never gets saved in the notebook or committed anywhere.

In [ ]:
!git clone --depth 1 https://github.com/NFAsylum/repo-rag.git
%cd repo-rag
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
import os

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

## Ingest

Clones the target repo, chunks its docs/code, embeds locally with `sentence-transformers`, and persists the index into ChromaDB. Here we index Babel's `docs/` and `examples/`; drop `--dirs` to index a whole repo, or change `--repo` for a different one.

In [ ]:
!python ingest.py --dirs docs,examples

## Query

Ask questions grounded in the indexed repo. Each answer lists the source files it was retrieved from.

In [ ]:
from query import query

sample_questions = [
    "What is Babel and what problem does it solve?",
    "Which programming languages does Babel support for translation?",
    "How do I add support for a new IDE to Babel?",
]

for q in sample_questions:
    result = query(q)
    print(f"Q: {q}\nA: {result['answer']}\nSources: {[s['file'] for s in result['sources']]}\n")

In [ ]:
# Try your own question
result = query("Your question here")
print(result["answer"])
print("Sources:", [s["file"] for s in result["sources"]])